## Data Description

### Who assembled this data, and why?

Data are provided by the Bonanza Creek LTER, a partnership between the University of Alaska Fairbanks, and the U.S. Forest Service. Significant funding for collection of these data was provided by the National Science Foundation Long-Term Ecological Research program


### What does a single row in the data represent?

One row is one hare observation 

### When and where were the observations collected?

5 locales in the Tanana valley, from Tok in the east to Clear in the west from 1999 to 2002. 

### Does the dataset contain sensitive data?
No

### What study protocols were used to collect the data?


### What are the units for the hares’ weight?
g

### Does the metadata say how missing data is encoded?
NaN

### Is there a publication associated with this dataset?

References:

Kielland, K., F.S. Chapin, R.W. Ruess, and Bonanza Creek LTER. 2017. Snowshoe hare physical data in Bonanza Creek Experimental Forest: 1999-Present ver 22. Environmental Data Initiative. https://doi.org/10.6073/pasta/03dce4856d79b91557d8e6ce2cbcdc14 (Accessed 2026-10-16).

![hare](https://thumb.wikimedia.org/wikipedia/commons/thumb/8/8a/SNOWSHOE_HARE_%28Lepus_americanus%29_%285-28-2015%29_quoddy_head%2C_washington_co%2C_maine_-01_%2818988734889%29.jpg/1920px-SNOWSHOE_HARE_%28Lepus_americanus%29_%285-28-2015%29_quoddy_head%2C_washington_co%2C_maine_-01_%2818988734889%29.jpg?utm_source=commons.wikimedia.org&utm_campaign=index&utm_content=thumbnail&_=20170313021652)

## Data loading and preliminary exploration

In [1]:
# Read in data
import numpy as np
import pandas as pd

#Import data
hares = pd.read_csv('data/55_Hare_Data_2012.txt')

In [2]:
# Check dimensions and dtypes
hares.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3380 entries, 0 to 3379
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        3380 non-null   object 
 1   time        264 non-null    object 
 2   grid        3380 non-null   object 
 3   trap        3368 non-null   object 
 4   l_ear       3332 non-null   object 
 5   r_ear       3211 non-null   object 
 6   sex         3028 non-null   object 
 7   age         1269 non-null   object 
 8   weight      2845 non-null   float64
 9   hindft      1633 non-null   float64
 10  notes       243 non-null    object 
 11  b_key       3333 non-null   float64
 12  session_id  3380 non-null   int64  
 13  study       3217 non-null   object 
dtypes: float64(3), int64(1), object(10)
memory usage: 369.8+ KB


In [4]:
# Check NAs
hares.isna().sum()

date             0
time          3116
grid             0
trap            12
l_ear           48
r_ear          169
sex            352
age           2111
weight         535
hindft        1747
notes         3137
b_key           47
session_id       0
study          163
dtype: int64

In [19]:
# Recapture distribution
captures_per_hare = hares.groupby('l_ear').size()

captures_per_hare.describe()

count    1020.000000
mean        3.266667
std         4.051371
min         1.000000
25%         1.000000
50%         2.000000
75%         4.000000
max        42.000000
dtype: float64

In [21]:
# Minimum and maximum values for the weight measurements
print(hares['weight'].max())
print(hares['weight'].min())
print("Weight(g):", "Min:", hares[
    'weight'].min() ,"-", "Max:", hares['weight'].max())

2365.0
0.0
Weight(g): Min: 0.0 - Max: 2365.0


In [8]:
# Minimum and maximum values for the hind feet measurements
print(hares['hindft'].max())
print(hares['hindft'].min())

160.0
60.0


In [10]:
# Unique values for categorical columns
hares['sex'].unique()

array([nan, 'M', 'F', '?', 'F?', 'M?', 'pf', 'm', 'f', 'f?', 'm?', 'f ',
       'm '], dtype=object)

In [13]:
hares['grid'].unique()

array(['bonrip', 'bonbs', 'Bonbs', 'bonmat', 'bonmat '], dtype=object)

In [15]:
hares['study'].unique()

array(['Population', 'Collar', nan, 'Metabolic', 'Metabolic/Collar'],
      dtype=object)

## Detecting messy values

Allowed values for hare sex according to metadata

| sex code      | Descroption |
| ----------- | ----------- |
| m      | male      |
| f   | female        |
| m?  | male not confirmed        |



In [23]:
# Check number of times each unique sex non-NA value appears
hares['sex'].value_counts()

sex
F     1161
M      730
f      556
m      515
?       40
F?      10
f        4
m        4
f?       3
M?       2
m?       2
pf       1
Name: count, dtype: int64

In [24]:
# Include NAs
hares['sex'].value_counts(dropna=False)

sex
F      1161
M       730
f       556
m       515
NaN     352
?        40
F?       10
f         4
m         4
f?        3
M?        2
m?        2
pf        1
Name: count, dtype: int64

### Do the values in the sex column correspond to the values declared in the metadata?
Not entirely.

### What could have been potential causes for multiple codes?
The data was encoded differently by different individuals.

### Are there seemingly repeated values? If so, what could be the cause?
There are seemingly repeated values. The difference may be due to spaces, case inconsistency, uncertain codes, etc.

## Clean values

In [28]:
# Remove spaces
hares['sex'].str.strip(' ').str.lower().value_counts(dropna=False)

sex
f      1721
m      1249
NaN     352
?        40
f?       13
m?        4
pf        1
Name: count, dtype: int64

In [29]:
hares['sex_simple'] = hares['sex'].str.strip(' ').str.lower()

In [30]:
hares['sex_simple'].value_counts(dropna=False)

sex_simple
f      1721
m      1249
NaN     352
?        40
f?       13
m?        4
pf        1
Name: count, dtype: int64

In [32]:
hares['sex_simple'] = hares['sex_simple'].replace('pf', 'f?')
hares['sex_simple'].value_counts(dropna=False)

sex_simple
f      1721
m      1249
NaN     352
?        40
f?       14
m?        4
Name: count, dtype: int64

In [34]:
hares['sex_simple'] = hares['sex_simple'].replace('?', np.NaN)
hares['sex_simple'].value_counts(dropna=False)

sex_simple
f      1721
m      1249
NaN     392
f?       14
m?        4
Name: count, dtype: int64

## Calculate mean weight

In [35]:
# Calculate mean weight (g) by sex
hares.groupby('sex_simple')['weight'].mean()

sex_simple
f     1365.164792
f?    1095.000000
m     1349.935542
m?    1446.666667
Name: weight, dtype: float64